# Sentiment Label Pipeline

Pipeline to produce sentence-level sentiment labels for MD&A text:

1. **Explode sentences** — load `mda_shared_preprocessed.csv` via PySpark and explode the `sentences` array into one row per sentence
2. **Manual labels** — placeholder for hand-labelling a seed set
3. **Few-shot labelling** — use a local Llama 3.2 model via Ollama (free, no API cost)
4. **Audit** — stratified sample of 1 000 rows for human review


## 0. Setup


In [8]:
!pip install ollama

In [ ]:
# pip install pyspark ollama pandas pyarrow openpyxl
import os, json, time
import pandas as pd
import ollama
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType

DATA_DIR   = Path("Datasets/interim")
OUTPUT_DIR = Path("Datasets/labelled")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_CSV          = DATA_DIR / "mda_shared_preprocessed.csv"
EXPLODED_PARQUET   = OUTPUT_DIR / "sentences_exploded.parquet"
SUBSET_PARQUET     = DATA_DIR  / "data_subset.parquet"
GOLD_XLSX          = DATA_DIR  / "gold_data.xlsx"
NON_GOLD_PARQUET   = DATA_DIR  / "non_gold_data.parquet"
LABELLED_PARQUET   = OUTPUT_DIR / "sentences_labelled.parquet"
AUDIT_CSV          = OUTPUT_DIR / "audit_sample.csv"

LABELS       = ["positive", "negative", "neutral"]
OLLAMA_MODEL = "llama3.2"   # must be pulled: `ollama pull llama3.2`
SEED         = 42
SUBSET_N     = 30_000
GOLD_N       = 800


## 1. Explode sentences with PySpark


In [ ]:
spark = (
    SparkSession.builder.appName("SentimentLabelPipeline")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

In [14]:
# --- load CSV -----------------------------------------------------------
raw = spark.read.csv(
    str(INPUT_CSV),
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"',
)

print(f"Rows: {raw.count():,}  |  Columns: {len(raw.columns)}")
raw.printSchema()

Rows: 18,183  |  Columns: 9
root
 |-- doc_id: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- filing_type: string (nullable = true)
 |-- filing_date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- quarter: string (nullable = true)
 |-- sentences: string (nullable = true)
 |-- clean_text: string (nullable = true)
 |-- n_sentences: integer (nullable = true)



In [ ]:
# The 'sentences' column is stored as a JSON string representing a list.
# Parse it into a Spark ArrayType then explode.

parse_sentences = F.udf(lambda s: json.loads(s) if s else [], ArrayType(StringType()))

exploded = (
    raw.withColumn("sentences_arr", parse_sentences(F.col("sentences")))
    .withColumn("sentence", F.explode("sentences_arr"))
    .withColumn("sentence_idx", F.monotonically_increasing_id())  # unique row id
    .select(
        F.col("doc_id"),
        F.col("company_name"),
        F.col("filing_type"),
        F.col("filing_date"),
        F.col("year"),
        F.col("quarter"),
        F.col("sentence_idx"),
        F.col("sentence"),
    )
    .filter(F.length(F.col("sentence")) > 20)  # drop near-empty sentences
)

print(f"Total sentences: {exploded.count():,}")
exploded.show(5, truncate=80)

Total sentences: 5,500,287
+-----------------------------+-------------+-----------+-----------+----+-------+------------+--------------------------------------------------------------------------------+
|                       doc_id| company_name|filing_type|filing_date|year|quarter|sentence_idx|                                                                        sentence|
+-----------------------------+-------------+-----------+-----------+----+-------+------------+--------------------------------------------------------------------------------+
|1-800-PetMeds_10-Q_2020-01-28|1-800-PetMeds|       10-Q| 2020-01-28|2020|     Q1|           0|executive summary petmed express was incorporated in the state of florida in ...|
|1-800-PetMeds_10-Q_2020-01-28|1-800-PetMeds|       10-Q| 2020-01-28|2020|     Q1|           1|the companys common stock is traded on the nasdaq global select market under ...|
|1-800-PetMeds_10-Q_2020-01-28|1-800-PetMeds|       10-Q| 2020-01-28|2020|     Q1|      

Traceback (most recent call last):
  File "/Users/lunlun/Downloads/Github/textmining_grp6/.venv/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
  File "/Users/lunlun/Downloads/Github/textmining_grp6/.venv/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
BrokenPipeError: [Errno 32] Broken pipe
Traceback (most recent call last):
  File "/Users/lunlun/Downloads/Github/textmining_grp6/.venv/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
  File "/Users/lunlun/Downloads/Github/textmining_grp6/.venv/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
BrokenPipeError: [Errno 32] Broken pipe


## 2. Manual Labelling of Subset


In [ ]:
# ── 2a. Sample 30k rows from 5.5M sentences ──────────────────────────────────
SUBSET_N = 30_000
GOLD_N = 800

SUBSET_PARQUET = DATA_DIR / "data_subset.parquet"
GOLD_XLSX = DATA_DIR / "gold_data.xlsx"

# Stratified sample across companies so no single company dominates
fraction = SUBSET_N / exploded.count()  # approximate fraction needed

data_subset_spark = exploded.sample(
    withReplacement=False, fraction=min(fraction * 1.05, 1.0), seed=SEED
).limit(SUBSET_N)

data_subset = data_subset_spark.toPandas()
print(f"data_subset shape: {data_subset.shape}")
print(data_subset["company_name"].nunique(), "unique companies")

# Persist for downstream use
data_subset.to_parquet(SUBSET_PARQUET, index=False)
print(f"Saved → {SUBSET_PARQUET}")


Traceback (most recent call last):                                  (0 + 1) / 1]
  File "/Users/lunlun/Downloads/Github/textmining_grp6/.venv/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
  File "/Users/lunlun/Downloads/Github/textmining_grp6/.venv/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
BrokenPipeError: [Errno 32] Broken pipe
Traceback (most recent call last):
  File "/Users/lunlun/Downloads/Github/textmining_grp6/.venv/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
  File "/Users/lunlun/Downloads/Github/textmining_grp6/.venv/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
BrokenPipeError: [Errno 32] Broken pipe


data_subset shape: (30000, 8)
448 unique companies
Saved → Datasets/interim/data_subset.parquet


In [ ]:
# ── 2b. Split data_subset → gold_data (800) + non_gold_data (29 200) ─────────
# Stratify by filing_type so 10-K and 10-Q are proportionally represented.

strata_col   = "filing_type"
strata_sizes = data_subset.groupby(strata_col).size()
total        = strata_sizes.sum()

gold_parts = []
for filing_type, count in strata_sizes.items():
    n       = max(1, round(GOLD_N * count / total))
    stratum = data_subset[data_subset[strata_col] == filing_type]
    gold_parts.append(stratum.sample(min(n, len(stratum)), random_state=SEED))

gold_data = (
    pd.concat(gold_parts)
    .drop_duplicates(subset="sentence_idx")
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
    .head(GOLD_N)
)

# Everything not in gold_data becomes non_gold_data
gold_idx      = set(gold_data["sentence_idx"])
non_gold_data = (
    data_subset[~data_subset["sentence_idx"].isin(gold_idx)]
    .reset_index(drop=True)
    .assign(label=None, label_source=None)   # columns expected by Section 3
)

# ── Save gold set as Excel for manual annotation ──────────────────────────────
gold_export = gold_data[[
    "sentence_idx", "doc_id", "company_name", "filing_type",
    "year", "quarter", "sentence",
]].assign(label="", notes="")

gold_export.to_excel(GOLD_XLSX, index=False)

# ── Save non_gold for few-shot labelling ──────────────────────────────────────
non_gold_data.to_parquet(NON_GOLD_PARQUET, index=False)

print(f"gold_data     : {len(gold_data):,} rows  →  {GOLD_XLSX}")
print(f"non_gold_data : {len(non_gold_data):,} rows  →  {NON_GOLD_PARQUET}")
print(f"\nfiling_type split in gold_data:")
print(gold_data["filing_type"].value_counts().to_string())

# df is the working frame for Section 3 (few-shot labelling)
df = non_gold_data.copy()


## 3. Few-shot labelling with Llama 3.2 (Ollama — local, free)

Uses the manually-labelled seed rows as few-shot examples embedded in the prompt.  
All remaining unlabelled sentences are sent to Llama in batches via Ollama.

> **Prerequisite:** `ollama pull llama3.2` (one-time download, ~2 GB)


In [ ]:
# Verify Ollama is running and the model is available
available = [m.model for m in ollama.list().models]
assert any(OLLAMA_MODEL in m for m in available), (
    f"Model '{OLLAMA_MODEL}' not found. Run: ollama pull {OLLAMA_MODEL}\n"
    f"Available: {available}"
)
print(f"Ollama ready — using model: {OLLAMA_MODEL}")

In [ ]:
# ── Few-shot examples ────────────────────────────────────────────────────────
# Drawn from the manual seed set (or hard-coded here as a fallback).

FEW_SHOT_EXAMPLES = [
    {
        "sentence": "net revenues increased by to million driven by strong demand across all product lines.",
        "label": "positive",
        "reason": "Revenue growth and strong demand indicate positive financial performance.",
    },
    {
        "sentence": "operating expenses rose sharply due to higher restructuring charges and asset impairments.",
        "label": "negative",
        "reason": "Rising costs from restructuring and impairments signal financial strain.",
    },
    {
        "sentence": "the company markets its products through national advertising campaigns.",
        "label": "neutral",
        "reason": "Factual description of business activity with no positive or negative signal.",
    },
    {
        "sentence": "gross margin expanded by basis points reflecting improved operational efficiency.",
        "label": "positive",
        "reason": "Margin expansion and efficiency gains are positive financial signals.",
    },
    {
        "sentence": "the company faces significant liquidity risks as cash reserves declined to million.",
        "label": "negative",
        "reason": "Liquidity risk and declining cash are negative financial signals.",
    },
    {
        "sentence": "condensed consolidated financial statements have been prepared in accordance with gaap.",
        "label": "neutral",
        "reason": "Boilerplate compliance statement with no directional financial signal.",
    },
]

# If manual labels exist, augment with them
if "manual" in df["label_source"].values:
    extra = (
        df[df.label_source == "manual"]
        .dropna(subset=["label"])
        .sample(min(10, df.label_source.eq("manual").sum()), random_state=SEED)[
            ["sentence", "label"]
        ]
        .assign(reason="Human-labelled example.")
        .to_dict(orient="records")
    )
    FEW_SHOT_EXAMPLES = extra + FEW_SHOT_EXAMPLES

print(f"Using {len(FEW_SHOT_EXAMPLES)} few-shot examples")

In [ ]:
def build_system_prompt(examples: list[dict]) -> str:
    examples_block = "\n".join(
        f'  Sentence: "{ex["sentence"]}"\n  Label: {ex["label"]}' for ex in examples
    )
    return (
        "You are a financial-text sentiment classifier specialising in SEC MD&A filings.\n\n"
        "Classify each sentence as exactly one of: positive | negative | neutral\n\n"
        "Guidelines:\n"
        "- positive  → revenue growth, margin expansion, improved outlook, cost savings\n"
        "- negative  → revenue decline, losses, impairments, liquidity risk, restructuring charges\n"
        "- neutral   → boilerplate disclosures, accounting policy statements, factual descriptions\n\n"
        f"Few-shot examples:\n{examples_block}\n\n"
        "You will receive a JSON array of objects with 'idx' and 'sentence' fields.\n"
        "Respond with a JSON array — one object per input sentence, preserving order:\n"
        '[{"idx": <idx>, "label": "<positive|negative|neutral>"}, ...]\n'
        "Output ONLY the JSON array. No explanation, no markdown fences."
    )


SYSTEM_PROMPT = build_system_prompt(FEW_SHOT_EXAMPLES)
print(SYSTEM_PROMPT[:400], "\n...")

In [ ]:
def label_batch(batch: pd.DataFrame, retries: int = 3) -> list[dict]:
    """Send a batch of sentences to Llama via Ollama and return label dicts."""
    user_content = json.dumps(
        [
            {"idx": int(row.sentence_idx), "sentence": row.sentence}
            for row in batch.itertuples()
        ],
        ensure_ascii=False,
    )
    for attempt in range(retries):
        try:
            response = ollama.chat(
                model=OLLAMA_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_content},
                ],
                options={"temperature": 0},  # deterministic
            )
            raw = response["message"]["content"].strip()
            # Strip markdown fences if the model adds them
            if raw.startswith("```"):
                raw = raw.split("```")[1]
                if raw.startswith("json"):
                    raw = raw[4:]
            results = json.loads(raw)
            for r in results:
                if r["label"] not in LABELS:
                    raise ValueError(f"Unexpected label: {r['label']!r}")
            return results
        except Exception as e:
            print(f"  Attempt {attempt + 1} failed: {e}")
            time.sleep(2**attempt)
    return []  # give up — rows stay unlabelled


print("label_batch() defined")

In [ ]:
BATCH_SIZE = 25  # sentences per API call (keep prompt small)

unlabelled = df[df["label"].isna()].copy()
print(f"Sentences to label: {len(unlabelled):,}")

results_acc = []

for batch_start in range(0, len(unlabelled), BATCH_SIZE):
    batch = unlabelled.iloc[batch_start : batch_start + BATCH_SIZE]
    results = label_batch(batch)
    results_acc.extend(results)

    if (batch_start // BATCH_SIZE + 1) % 20 == 0:
        done = batch_start + len(batch)
        print(
            f"  {done:,}/{len(unlabelled):,} labelled ({done / len(unlabelled) * 100:.1f}%)"
        )

print(f"Done — {len(results_acc):,} labels received")

In [ ]:
# Merge few-shot labels back into df
fs_map = {r["idx"]: r["label"] for r in results_acc}
mask = df["sentence_idx"].isin(fs_map)
df.loc[mask, "label"] = df.loc[mask, "sentence_idx"].map(fs_map)
df.loc[mask, "label_source"] = "few_shot"

print("Label distribution:")
print(df["label"].value_counts())
print(f"\nStill unlabelled: {df['label'].isna().sum():,}")

In [ ]:
# Save few-shot labelled non_gold_data
non_gold_data = df.copy()
non_gold_data.to_parquet(NON_GOLD_PARQUET, index=False)
df.to_parquet(LABELLED_PARQUET, index=False)

print(f"non_gold_data (labelled) → {NON_GOLD_PARQUET}")
print(f"sentences_labelled       → {LABELLED_PARQUET}")
print(f"\nLabel distribution:\n{non_gold_data['label'].value_counts()}")
print(f"Unlabelled: {non_gold_data['label'].isna().sum():,}")


## 4. Audit — stratified sample of 1 000 rows

Stratify by **label × filing_type** so each stratum is represented proportionally.  
The exported CSV can be reviewed in a spreadsheet; add an `audit_label` column and flag disagreements.


In [ ]:
AUDIT_N = 1_000

labelled_df = pd.read_parquet(LABELLED_PARQUET)
labelled_df = labelled_df.dropna(subset=["label"])

# Stratify by label × filing_type
strata_col = ["label", "filing_type"]
strata_sizes = labelled_df.groupby(strata_col).size()
total = strata_sizes.sum()

audit_parts = []
for keys, count in strata_sizes.items():
    n = max(1, round(AUDIT_N * count / total))
    subset = labelled_df[labelled_df[strata_col].apply(tuple, axis=1) == keys]
    audit_parts.append(subset.sample(min(n, len(subset)), random_state=SEED))

audit_df = (
    pd.concat(audit_parts)
    .drop_duplicates(subset="sentence_idx")
    .sample(frac=1, random_state=SEED)  # shuffle rows
    .reset_index(drop=True)
    .head(AUDIT_N)
)

print(f"Audit sample size: {len(audit_df):,}")
print(audit_df.groupby(["label", "filing_type"]).size().unstack(fill_value=0))

In [ ]:
# Add empty audit columns for human review
audit_df["audit_label"] = ""  # reviewer fills in: positive / negative / neutral
audit_df["audit_correct"] = ""  # reviewer fills in: yes / no
audit_df["audit_notes"] = ""

audit_out = audit_df[
    [
        "sentence_idx",
        "doc_id",
        "company_name",
        "filing_type",
        "year",
        "quarter",
        "sentence",
        "label",
        "label_source",
        "audit_label",
        "audit_correct",
        "audit_notes",
    ]
]

audit_out.to_csv(AUDIT_CSV, index=False)
print(f"Audit CSV saved → {AUDIT_CSV}")

In [ ]:
# ── Post-audit summary  (run after filling in audit_correct) ─────────────────
# Uncomment once the audit CSV has been reviewed.

# reviewed = pd.read_csv(AUDIT_CSV)
# reviewed = reviewed[reviewed["audit_correct"].isin(["yes", "no"])]
#
# accuracy_overall = reviewed["audit_correct"].eq("yes").mean()
# print(f"Overall audit accuracy: {accuracy_overall:.2%} ({len(reviewed):,} reviewed rows)")
#
# print("\nAccuracy by label:")
# print(reviewed.groupby("label")["audit_correct"].apply(lambda s: s.eq("yes").mean()).round(3))
#
# print("\nAccuracy by filing_type:")
# print(reviewed.groupby("filing_type")["audit_correct"].apply(lambda s: s.eq("yes").mean()).round(3))
#
# disagreements = reviewed[reviewed.audit_correct == "no"]
# print(f"\nDisagreements: {len(disagreements):,}")
# display(disagreements[["sentence", "label", "audit_label", "audit_notes"]].head(20))